# Can the mtanh baseline fit be trusted at the pedestal?

It fits, and measures the fit. No scaling until the final section, which
applies it deliberately to answer a different question: how much of the
space the axes can actually reach.

**Why it exists.** The sparse-grid axes are scale factors on mtanh fit
parameters, so a bad fit does not produce a bad point — it redefines what every
axis value means, for the whole scan, silently. A first pass flagged
`rms_relative` at 2.7–3.0% on 132588 and 11.8% on 129038's `ne`, against 0.3–0.6%
on 132543.

**But that number is the wrong test**, and this notebook exists to replace it.
`rms_relative` is computed over the *whole* profile, core to separatrix. A fit
can score badly because it misses the core — which the scan never touches — and
still be excellent where the pedestal axes act. The reverse is also possible: a
respectable global number hiding a bad pedestal.

So: error as a function of radius, and a total taken only over the pedestal
window. That is the number that licenses the scale factors.

**Two parameterizations, both fitted.**

- `fit_mtanh` — Bruncrona et al. 2025 Eq. 4, a *pedestal-only* form. What
  `apply_mtanh_ped` uses, so its pedestal error is the one that governs the
  current axes.
- `fit_mtanh_full` — Stefanikova 2016, axis-to-SOL in one formula. What
  `apply_mtanh_full` would use. Included because if the pedestal-only form is
  the thing failing, the global form is the ready alternative.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

from TPED.projects.discharge_tools.src.discharge_data import DischargeData
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import (
    fit_mtanh, fit_mtanh_full)

DISCHARGE_ROOT = r"C:/Users/joesc/git/ST_research/NSTXU_discharges"

# 129038's directory holds five pfiles, so auto-discovery refuses to guess and
# needs an explicit one. Everything else auto-discovers.
DISCHARGES = {
    129015: {},
    129038: {"pfile": "p129038.00400"},
    132543: {},
    132588: {},
}

# Profiles the scan axes act on, plus the ones that follow through
# quasineutrality — a fit that is fine for ne and wrong for ni still matters,
# because CHEASE sees both.
VARS = ["Te", "Ti", "ne", "ni"]

# The window the verdict is taken over. Inside it the fit governs the axes;
# outside, error is real but does not touch the scan. 0.95 rather than 1.0
# deliberately: the last few points are separatrix/SOL, where the pfile itself
# is least trustworthy and where nothing is scanned.
PED_WINDOW = (0.60, 0.95)

# Weighting used for the pedestal fit throughout, matching the campaign.
FIT_KWARGS = dict(pedestal_weight=8.0)

# Where GENE is actually run, per discharge. These matter more than they look:
# 132543/132588's radii are the q=4 and q=5 surfaces, and both sit INSIDE the
# pedestal top (psi_mid is 0.92-0.95 with delta_ped ~0.10-0.12, so the pedestal
# spans roughly 0.87-0.98). The fit therefore has to be good at 0.70-0.98, not
# merely at the pedestal -- a point the verdict window alone would hide.
ANALYSIS_RADII = {129015: (0.85,), 129038: (0.85,),
                  132543: (0.736, 0.825), 132588: (0.736, 0.825)}

# Scale box the campaign currently uses, for the coverage section.
SCALE_BOX = (0.7, 1.3)

PED_LO, PED_HI = PED_WINDOW
print(f"verdict window: rho_tor {PED_LO} to {PED_HI}")

## Fit every profile of every discharge, both parameterizations

`fit_mtanh_full` is allowed to fail without taking the notebook with it — the
Stefanikova form has more parameters and does not always converge on an ST
profile. A failure there is a result, not an error.

In [ ]:
def load(shot):
    d = os.path.join(DISCHARGE_ROOT, str(shot))
    kw = {"input_dir": d}
    name = DISCHARGES[shot].get("pfile")
    if name:
        kw["pfile"] = os.path.join(d, name)
    return DischargeData(**kw)


results = {}
for shot in DISCHARGES:
    phys = DischargePhysics(load(shot))
    x = np.asarray(phys.rhot.values)
    entry = {"x": x, "vars": {}}
    for var in VARS:
        if var not in phys.ds:
            continue
        y = np.asarray(getattr(phys, var).values)
        # Normalize the error by the profile RANGE, not by y itself: a relative
        # error against a value that goes to ~0 at the separatrix explodes there
        # and would swamp the pedestal, which is the region under test.
        scale = float(np.max(y) - np.min(y)) or 1.0
        rec = {"y": y, "scale": scale}
        for label, fn in (("ped", fit_mtanh), ("full", fit_mtanh_full)):
            try:
                profile, meta = fn(phys.ds, var, **FIT_KWARGS)
                yhat = np.asarray(profile(x), dtype=float)
                rec[label] = {"yhat": yhat,
                              "err": (yhat - y) / scale,
                              "rms_global": meta["rms_relative"],
                              "params": meta["fit_params"]}
            except Exception as exc:
                rec[label] = {"error": f"{type(exc).__name__}: {exc}"}
                print(f"  {shot} {var} {label}: FAILED {rec[label]['error']}")
        entry["vars"][var] = rec
    results[shot] = entry
    print(f"{shot}: fitted {list(entry['vars'])}")


def window_stats(x, err):
    """rms and max |error| inside the verdict window, as a fraction of range."""
    m = (x >= PED_LO) & (x <= PED_HI)
    e = err[m]
    return (float(np.sqrt(np.mean(e ** 2))), float(np.max(np.abs(e))))


TRUST = 0.01     # 1% of profile range, rms, inside the window

shots = sorted(results)
varlist = [v for v in VARS if any(v in e["vars"] for e in results.values())]

## Inspect the fits by eye — pedestal region

Rows are profiles, columns are discharges. Grey dots are the pfile data, solid
is the pedestal-only (Bruncrona) form, dashed is the full (Stefanikova) form.
The shaded band is the verdict window.

This is the plot to look at first. A number says a fit is bad; only this says
*how* — missed pedestal position, wrong width, a shoulder the form cannot
represent, or the fit having latched onto the core instead.

In [ ]:
def fit_grid(xlim, title, ylim_frac=None):
    varlist = [v for v in VARS if any(v in e["vars"] for e in results.values())]
    shots = sorted(results)
    fig, axes = plt.subplots(len(varlist), len(shots),
                             figsize=(3.6 * len(shots), 2.7 * len(varlist)),
                             squeeze=False)
    for i, var in enumerate(varlist):
        for j, shot in enumerate(shots):
            ax = axes[i][j]
            entry = results[shot]
            rec = entry["vars"].get(var)
            if rec is None:
                ax.set_visible(False)
                continue
            x, y = entry["x"], rec["y"]
            m = (x >= xlim[0]) & (x <= xlim[1])
            ax.plot(x[m], y[m], ".", ms=2.5, color="0.55", label="data", zorder=1)
            for label, style, col in (("ped", "-", "tab:red"),
                                      ("full", "--", "tab:blue")):
                f = rec.get(label, {})
                if "yhat" in f:
                    ax.plot(x[m], f["yhat"][m], style, lw=1.4, color=col,
                            label=label, zorder=2)
            ax.axvspan(PED_LO, PED_HI, color="tab:green", alpha=0.10, zorder=0)
            for b in (PED_LO, PED_HI):
                ax.axvline(b, color="tab:green", lw=0.8, alpha=0.6, zorder=0)
            # rms inside the window, printed on the panel so the picture and the
            # verdict are never read apart from each other
            fped = rec.get("ped", {})
            if "err" in fped:
                r = window_stats(x, fped["err"])[0]
                ax.text(0.03, 0.06, f"ped rms {r*100:.2f}%", fontsize=7,
                        transform=ax.transAxes,
                        color="tab:red" if r > TRUST else "green")
            ax.set_xlim(*xlim)
            if i == 0:
                ax.set_title(str(shot), fontsize=10)
            if j == 0:
                ax.set_ylabel(var)
            if i == len(varlist) - 1:
                ax.set_xlabel("rho_tor")
            ax.tick_params(labelsize=7)
    axes[0][-1].legend(fontsize=6)
    fig.suptitle(title)
    plt.tight_layout()
    return fig


fit_grid((0.60, 1.00), "mtanh fits, pedestal region — grey: data, red: pedestal form, blue: full form")
plt.show()

Same fits over the whole radius. A form that tracks the pedestal but
diverges in the core is still usable here — no scan axis touches the core — and
this is where that gets confirmed rather than assumed.

In [ ]:
fit_grid((0.0, 1.00), "mtanh fits, full radius")
plt.show()

## Error against radius — core to separatrix

One panel per discharge, every profile on it, both parameterizations
(pedestal-only solid, full dashed). The shaded band is the verdict window.

What to look for: error that is large in the core and small inside the band is
*fine* — the scan does not touch the core. Error that rises inside the band is
what disqualifies a fit.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5.0 * len(results), 4.2),
                         sharey=True, squeeze=False)
colors = {"Te": "tab:red", "Ti": "tab:orange", "ne": "tab:blue", "ni": "tab:cyan"}

for ax, (shot, entry) in zip(axes[0], sorted(results.items())):
    x = entry["x"]
    for var, rec in entry["vars"].items():
        for label, style in (("ped", "-"), ("full", "--")):
            f = rec.get(label, {})
            if "err" not in f:
                continue
            ax.plot(x, 100 * f["err"], style, lw=1.1, color=colors.get(var),
                    label=f"{var} {label}", alpha=0.9)
    ax.axvspan(PED_LO, PED_HI, color="k", alpha=0.07)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xlim(0, 1.0); ax.set_ylim(-15, 15)
    ax.set_xlabel("rho_tor"); ax.set_title(str(shot))
axes[0][0].set_ylabel("(fit - data) / range   [%]")
axes[0][-1].legend(fontsize=6, ncol=2)
fig.suptitle("mtanh fit error vs radius — solid: pedestal form, dashed: full form"
             f"   (shaded = verdict window {PED_LO}-{PED_HI})")
plt.tight_layout(); plt.show()

## Where does the trustworthy region start?

The panels above show every failure sitting on the **inner** side of the window
— 0.60–0.80 — while the pedestal proper is tracked well everywhere. That makes
the window boundary a real choice rather than a detail: a window that reaches
into the core is asking a pedestal form to fit core structure it was never meant
to represent.

This sweeps the inner edge and reports rms for each, so the trustworthy region
is measured rather than assumed. Read it as: how far in can the fit be believed,
per discharge and profile.

In [ ]:
inner_edges = [0.60, 0.70, 0.75, 0.80, 0.85]

fig, axes = plt.subplots(1, len(varlist), figsize=(3.6 * len(varlist), 3.4),
                         sharey=True, squeeze=False)
for ax, var in zip(axes[0], varlist):
    for shot in shots:
        rec = results[shot]["vars"].get(var, {})
        if "err" not in rec.get("ped", {}):
            continue
        x, err = results[shot]["x"], rec["ped"]["err"]
        vals = []
        for lo in inner_edges:
            m = (x >= lo) & (x <= PED_HI)
            vals.append(100 * float(np.sqrt(np.mean(err[m] ** 2))))
        ax.plot(inner_edges, vals, "o-", ms=4, lw=1.2, label=str(shot))
    ax.axhline(100 * TRUST, color="k", ls="--", lw=1.0)
    ax.set_xlabel("window inner edge (rho_tor)")
    ax.set_title(var); ax.set_yscale("log")
axes[0][0].set_ylabel(f"rms to {PED_HI}  [% of range]")
axes[0][-1].legend(fontsize=7)
fig.suptitle("pedestal-form error vs how far inward the window reaches "
             "— dashed line is the trust threshold")
plt.tight_layout(); plt.show()

print(f"{'shot':>7} {'var':<4} " + " ".join(f"{e:>7.2f}" for e in inner_edges))
print("-" * (12 + 8 * len(inner_edges)))
for shot in shots:
    for var in varlist:
        rec = results[shot]["vars"].get(var, {})
        if "err" not in rec.get("ped", {}):
            continue
        x, err = results[shot]["x"], rec["ped"]["err"]
        cells = []
        for lo in inner_edges:
            m = (x >= lo) & (x <= PED_HI)
            cells.append(f"{100*float(np.sqrt(np.mean(err[m]**2))):>6.2f}%")
        print(f"{shot:>7} {var:<4} " + " ".join(cells))
print(); print("A row that drops below the threshold as the edge moves outward is a "
      "fit that is sound at the pedestal and poor in the core -- which is "
      "acceptable, since no scan axis acts there. A row that stays high at 0.85 "
      "is a genuinely bad pedestal fit.")

## The verdict, as a picture

Pedestal-window rms per profile, both forms, with the trust threshold drawn.
Bars below the line are usable; above it are not. This is the summary — the
table below it is the same numbers for the record.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
labels, ped_vals, full_vals = [], [], []
for shot in shots:
    for var in varlist:
        rec = results[shot]["vars"].get(var, {})
        x = results[shot]["x"]
        labels.append(f"{shot} {var}")
        ped_vals.append(100 * window_stats(x, rec["ped"]["err"])[0]
                        if "err" in rec.get("ped", {}) else np.nan)
        full_vals.append(100 * window_stats(x, rec["full"]["err"])[0]
                         if "err" in rec.get("full", {}) else np.nan)

idx = np.arange(len(labels))
ax.bar(idx - 0.2, ped_vals, 0.4, label="pedestal form", color="tab:red")
ax.bar(idx + 0.2, full_vals, 0.4, label="full form", color="tab:blue")
ax.axhline(100 * TRUST, color="k", ls="--", lw=1.2,
           label=f"trust threshold {TRUST:.0%}")
ax.set_xticks(idx); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel(f"rms error in {PED_LO}-{PED_HI}  [% of range]")
ax.set_yscale("log")
ax.legend(fontsize=8)
ax.set_title("pedestal-window fit error — below the dashed line is usable")
plt.tight_layout(); plt.show()

## The verdict — error inside the pedestal window only

`rms_ped` and `max_ped` are taken over the window alone. `rms_global` is the
whole-profile number for comparison: where the two diverge, the global number
was measuring something the scan does not care about.

In [ ]:
rows = []
for shot, entry in sorted(results.items()):
    x = entry["x"]
    for var, rec in entry["vars"].items():
        for label in ("ped", "full"):
            f = rec.get(label, {})
            if "err" not in f:
                rows.append({"shot": shot, "var": var, "form": label,
                             "rms_ped": float("nan"), "max_ped": float("nan"),
                             "rms_global": float("nan"), "ok": False,
                             "note": f.get("error", "")[:40]})
                continue
            rms, mx = window_stats(x, f["err"])
            rows.append({"shot": shot, "var": var, "form": label,
                         "rms_ped": rms, "max_ped": mx,
                         "rms_global": f["rms_global"],
                         "ok": rms <= TRUST, "note": ""})

hdr = (f"{'shot':>7} {'var':<4} {'form':<5} {'rms_ped':>9} {'max_ped':>9} "
       f"{'rms_global':>11} {'verdict':>9}")
print(hdr); print("-" * len(hdr))
for r in rows:
    v = "TRUST" if r["ok"] else "reject"
    rp = f"{r['rms_ped']*100:>8.2f}%" if r["rms_ped"] == r["rms_ped"] else f"{'--':>9}"
    mp = f"{r['max_ped']*100:>8.2f}%" if r["max_ped"] == r["max_ped"] else f"{'--':>9}"
    rg = f"{r['rms_global']*100:>10.2f}%" if r["rms_global"] == r["rms_global"] else f"{'--':>11}"
    print(f"{r['shot']:>7} {r['var']:<4} {r['form']:<5} {rp} {mp} {rg} {v:>9}"
          + (f"   {r['note']}" if r["note"] else ""))
print(f"\nTRUST threshold: rms inside {PED_LO}-{PED_HI} at or below "
      f"{TRUST:.0%} of profile range")

### Per-discharge summary for the axes actually in use

The scan axes are `Te_ped_scale` and `ne_ped_scale`, both driving
`apply_mtanh_ped` — so the **pedestal form's** `Te` and `ne` rows are what
license them. `Ti`/`ni` follow through quasineutrality and the closure, so a bad
fit there reaches CHEASE even though no axis names it.

In [ ]:
print(f"{'shot':>7}  {'Te ped':>8} {'ne ped':>8}   axes usable?")
print("-" * 48)
verdict = {}
for shot, entry in sorted(results.items()):
    x = entry["x"]
    vals = {}
    for var in ("Te", "ne"):
        f = entry["vars"].get(var, {}).get("ped", {})
        vals[var] = window_stats(x, f["err"])[0] if "err" in f else float("nan")
    ok = all(v == v and v <= TRUST for v in vals.values())
    verdict[shot] = {"Te_rms_ped": vals["Te"], "ne_rms_ped": vals["ne"], "ok": ok}
    print(f"{shot:>7}  {vals['Te']*100:>7.2f}% {vals['ne']*100:>7.2f}%   "
          f"{'yes' if ok else 'NO — refit before scaling'}")

out = "mtanh_fit_quality.json"
with open(out, "w") as f:
    json.dump({"ped_window": [PED_LO, PED_HI], "trust_threshold": TRUST,
               "rows": [{k: v for k, v in r.items()} for r in rows],
               "verdict": verdict}, f, indent=1, default=str)
print(f"\nwritten: {out}")

## If a fit is rejected — what to try

In order of effort. Re-run the cells above after each; the verdict table is the
arbiter.

1. **Raise `pedestal_weight`.** Currently 8. It upweights points inside
   `ped_threshold`–`edge_threshold`, buying pedestal accuracy at the core's
   expense — which is the trade this window says we want.
2. **Narrow `ped_threshold`.** Default 0.85. A fit whose `psi_mid` lands exactly
   on 0.85 has hit that bound rather than found the pedestal; moving the
   boundary lets it search where the pedestal actually is.
3. **Supply explicit `p0`.** Eight parameters — `[y_sep, a_y0, delta_ped,
   psi_mid, a_y1, psi_ped, alpha_1, alpha_2]`. Seed `psi_mid` and `delta_ped`
   from the profile by eye; those two are what the auto-guess gets wrong.
4. **Switch that discharge to `apply_mtanh_full`.** If the full form's pedestal
   error beats the pedestal form's in the table above, the axis should be
   defined on it instead — at the cost of a different, larger parameter set.

In [ ]:
for shot, entry in sorted(results.items()):
    x = entry["x"]
    for var in ("Te", "ne"):
        rec = entry["vars"].get(var, {})
        p, fl = rec.get("ped", {}), rec.get("full", {})
        if "err" not in p:
            continue
        rp = window_stats(x, p["err"])[0]
        if rp <= TRUST:
            continue
        params = p["params"]
        print(f"{shot} {var}: rms_ped {rp*100:.2f}%  "
              f"psi_mid {params.get('psi_mid'):.4f}  "
              f"delta_ped {params.get('delta_ped'):.4f}")
        if abs(params.get("psi_mid", 0) - 0.85) < 1e-6:
            print("    psi_mid is ON the ped_threshold bound — try step 2 first")
        if "err" in fl:
            rf = window_stats(x, fl["err"])[0]
            better = "BETTER" if rf < rp else "no better"
            print(f"    full form rms_ped {rf*100:.2f}% — {better}")

## What span do the axes actually buy? — a/L at the analysis radius

The only quantity that reaches the physics is what GENE computes from the
profiles it is handed, and for drive that is the normalized inverse scale
length. Fit parameters are not comparable between the two forms — they name
different things — so this measures the derived quantity instead, in absolute
units, across the scale box.

Both forms reproduce the profile to within a fraction of a percent, but they
distribute a height change differently in radius: the Bruncrona a_y0 term
saturates inward, so scaling it shifts the inner region almost rigidly and the
gradient barely moves, while Stefanikova's b_height pivots against a fixed
on-axis a_height, tilting the profile so the gradient moves with it. Same
curve, different derivative response — which is invisible in the fit overlays
and decisive for the scan.

a/L_pe is the KBM-relevant one, since that drive is the pressure gradient.

In [ ]:
def a_over_L(x, y, x0):
    i = np.argmin(np.abs(x - x0))
    return float(-np.gradient(y, x)[i] / y[i])


def scaled(phys, var, fit, kind, s):
    fn = phys.apply_mtanh_ped if kind == "ped" else phys.apply_mtanh_full
    return fn(var, fit=fit, scale_height=s,
              enforce_quasineutrality=True, qz=6.0)


scales = np.linspace(SCALE_BOX[0], SCALE_BOX[1], 7)
span_rows, curves = [], {}

for shot in shots:
    entry = results[shot]
    x = entry["x"]
    phys = DischargePhysics(load(shot))
    for x0 in ANALYSIS_RADII[shot]:
        for kind in ("ped", "full"):
            for axis in ("Te", "ne"):
                rec = entry["vars"].get(axis, {})
                if "params" not in rec.get(kind, {}):
                    continue
                fit = (fit_mtanh if kind == "ped" else fit_mtanh_full)(
                    phys.ds, axis, **FIT_KWARGS)[0]
                vals = {"Te": [], "ne": [], "pe": []}
                for s in scales:
                    q = scaled(phys, axis, fit, kind, s)
                    te = np.asarray(q.Te.values)
                    ne = np.asarray(q.ne.values)
                    vals["Te"].append(a_over_L(x, te, x0))
                    vals["ne"].append(a_over_L(x, ne, x0))
                    vals["pe"].append(a_over_L(x, ne * te, x0))
                base = vals["pe"][len(scales) // 2]
                span = max(vals["pe"]) - min(vals["pe"])
                span_rows.append({"shot": shot, "x0": x0, "form": kind,
                                  "axis": axis, "aLpe_lo": vals["pe"][0],
                                  "aLpe_hi": vals["pe"][-1], "span": span,
                                  "span_pct": 100 * span / abs(base)})
                curves[(shot, x0, kind, axis)] = vals["pe"]

hdr = (f"{'shot':>7} {'x0':>6} {'axis':<4} {'ped span':>10} {'full span':>10} "
       f"{'ped %':>7} {'full %':>7}")
print("a/L_pe span across the scale box "
      f"{SCALE_BOX[0]}-{SCALE_BOX[1]}")
print(hdr); print("-" * len(hdr))
for shot in shots:
    for x0 in ANALYSIS_RADII[shot]:
        for axis in ("Te", "ne"):
            p = next((r for r in span_rows if r["shot"] == shot and r["x0"] == x0
                      and r["form"] == "ped" and r["axis"] == axis), None)
            f = next((r for r in span_rows if r["shot"] == shot and r["x0"] == x0
                      and r["form"] == "full" and r["axis"] == axis), None)
            if not (p and f):
                continue
            print(f"{shot:>7} {x0:>6.3f} {axis:<4} {p['span']:>10.3f} "
                  f"{f['span']:>10.3f} {p['span_pct']:>6.1f}% {f['span_pct']:>6.1f}%")
print()
print("A span near zero means the axis cannot move the drive at that radius -- "
      "the scan is a no-op there whatever the fit quality. On 132543 at 0.825 "
      "the pedestal form moves a/L_pe under 2% across the WHOLE box.")

# The exactly-zero case has a clean cause worth testing for directly.
print()
print("pedestal floor (b_sol) -- a floor at zero makes scale_height a PURE")
print("MULTIPLIER of the pedestal component, and a/L is invariant under")
print("multiplication, so the axis moves the value but not the drive:")
for shot in shots:
    phys = DischargePhysics(load(shot))
    for axis in ("Te", "ne"):
        try:
            _, m = fit_mtanh_full(phys.ds, axis, **FIT_KWARGS)
            bs, bh = m["fit_params"]["b_sol"], m["fit_params"]["b_height"]
            flag = "   <-- cannot move a/L" if abs(bs) < 1e-9 else ""
            print(f"  {shot} {axis:<3} b_sol {bs:>12.4g} / b_height {bh:>10.4g}{flag}")
        except Exception:
            pass

Plotted. A flat line is an axis that cannot reach the space; slope and
range are what the scan buys. Non-monotonicity would be a separate problem —
the sparse grid assumes the QoI is smooth in the axes.

In [ ]:
pairs = [(shot, x0) for shot in shots for x0 in ANALYSIS_RADII[shot]]
fig, axes = plt.subplots(1, len(pairs), figsize=(3.3 * len(pairs), 3.4),
                         squeeze=False)
for ax, (shot, x0) in zip(axes[0], pairs):
    for axis, base_col in (("Te", "tab:red"), ("ne", "tab:blue")):
        for kind, style in (("ped", "-"), ("full", "--")):
            y = curves.get((shot, x0, kind, axis))
            if y is None:
                continue
            ax.plot(scales, y, style, color=base_col, lw=1.3,
                    label=f"{axis} {kind}")
    ax.set_title(f"{shot}  x0={x0}", fontsize=9)
    ax.set_xlabel("scale_height")
    ax.tick_params(labelsize=7)
axes[0][0].set_ylabel("a/L_pe")
axes[0][-1].legend(fontsize=6)
fig.suptitle("KBM-relevant drive reachable by each axis — flat means unreachable")
plt.tight_layout(); plt.show()

## Absolute-unit bounds — Boyle 2011, converted per discharge

Everything above is diagnosis. This is the handover: take the survey bounds in
the units they are quoted in, convert them to the scale factors this discharge's
fit needs, and check the target is actually reachable before a campaign spends
CHEASE time discovering it is not.

Bounds are stated in physical units because that is how the literature states
them and how Paper 1 has to report them. Scale factors are an implementation
detail of the transform, they differ per discharge and per variable, and quoting
a scan box in them is how the last three rounds of confusion started.

Widths are from Boyle 2011 PPCF (lithium scan, Fig. 7), which separates two
regimes — they are different plasmas, not a single range, and are scanned as
separate boxes rather than one wide one.

In [ ]:
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import fit_mtanh

# Boyle 2011 PPCF, Fig. 7: pedestal FULL widths in % psi_N.
#   panel 7a = Delta_ne, 7d = Delta_Te, 7g = Delta_pe
BOYLE_WIDTHS = {
    "ELMy":     {"dne": (6, 12), "dTe": (4, 7),  "dpe": (4, 8)},    # black symbols
    "ELM_free": {"dne": (14, 22), "dTe": (7, 10), "dpe": (8, 12)},  # blue diamonds
}

# Pedestal-top values and the ion/electron ratio. These set beta together with
# the widths, which is why they are frozen as one box rather than separately.
TARGETS = {
    "Te_ped": (0.2, 0.8),      # keV,  at B0 ~ 0.4 T
    "ne_ped": (3.0, 7.0),      # 1e19 m^-3
    "Ti_Te":  (1.0, 2.0),      # pedestal-top ratio
}

REGIME = "ELMy"                # which Boyle box the width axes use

# A required scale factor outside this is treated as out of reach rather than
# quietly emitted: nothing near it has been tested against cheaseBS, and the
# reshape limit is not yet established.
SCALE_SANITY = (0.3, 3.0)

print(f"regime {REGIME}: widths {BOYLE_WIDTHS[REGIME]}")
print(f"targets {TARGETS}")

### Measure each axis's response, then invert it

One probe per axis. The scale→metric mapping is linear to machine precision
(verified earlier, slope std 0.00000), so two points determine it exactly and
`scale = 1 + (target/nominal - 1)/slope` inverts it without iteration.

The metric differs by axis and is the physical quantity the bound is quoted in:
pedestal-top value for the heights, full width in %ψ_N for the widths, and the
pedestal-top ratio for `Ti_Te`.

In [ ]:
def _ped_top(phys, var):
    x = np.asarray(phys.rhot.values)
    m = (x >= 0.90) & (x <= 0.98)
    return float(np.max(np.asarray(getattr(phys, var).values)[m]))


def _width_psin(phys, var, fits):
    """Full pedestal width in % psi_N.

    Boyle quotes widths in poloidal flux; the fit works in rho_tor. Converting
    through the discharge's own rhot/rhop mapping is the only honest way to
    compare, and the factor is not a constant -- it depends where the pedestal
    sits.
    """
    rhot = np.asarray(phys.rhot.values)
    rhop = np.asarray(phys.rhop.values)
    _, meta = fit_mtanh(phys.ds, var, **FIT_KWARGS)
    fp = meta["fit_params"]
    lo, hi = fp["psi_mid"] - fp["delta_ped"] / 2, fp["psi_mid"] + fp["delta_ped"] / 2
    pl, ph = np.interp([lo, hi], rhot, rhop)
    return 100.0 * (ph ** 2 - pl ** 2)


# axis name -> (variable, transform kwarg, metric, unit, scale for reporting)
AXES = {
    "Te_ped_scale":   ("Te", "scale_height", "ped_top",  "keV",  1e-3),
    "ne_ped_scale":   ("ne", "scale_height", "ped_top",  "1e19", 1e-19),
    "Te_width_scale": ("Te", "scale_width",  "width",    "%psiN", 1.0),
    "ne_width_scale": ("ne", "scale_width",  "width",    "%psiN", 1.0),
    "Ti_Te_scale":    ("Ti", "scale_height", "ti_te",    "-",     1.0),
}


def metric_of(phys, var, kind, fits):
    if kind == "ped_top":
        return _ped_top(phys, var)
    if kind == "width":
        return _width_psin(phys, var, fits)
    if kind == "ti_te":
        return _ped_top(phys, "Ti") / _ped_top(phys, "Te")
    raise ValueError(kind)


def axis_response(phys, axis, fits, probe=1.2):
    """Nominal metric and its linear slope in the scale factor."""
    var, kwarg, kind, unit, disp = AXES[axis]
    fit = fits[var]

    def at(s):
        q = phys.apply_mtanh_ped(var, fit=fit, **{kwarg: s},
                                 enforce_quasineutrality=True, qz=6.0)
        return metric_of(q, var, kind, fits)

    m0 = at(1.0)
    slope = (at(probe) - m0) / m0 / (probe - 1.0)
    # Return the nominal in the SAME units the bounds are quoted in. The raw
    # dataset is eV and m^-3 while the targets are keV and 1e19, so inverting
    # against the raw value silently produces a meaningless scale factor. The
    # slope is a relative quantity and is unaffected by the conversion.
    return m0 * disp, slope, unit, disp


def scale_for(m0, slope, target):
    """Invert the linear map. Returns nan when the axis cannot move the metric."""
    if not np.isfinite(slope) or abs(slope) < 1e-6:
        return float("nan")
    return 1.0 + ((target / m0) - 1.0) / slope


print("axes:", ", ".join(AXES))

### Nominal values against the target box

`nominal` is where each discharge already sits. `need lo` / `need hi` are the
scale factors that would reach the bound. A nominal already inside its target
range is the comfortable case; one outside means the axis has to travel in one
direction only.

In [ ]:
bounds_per_shot, reach_rows = {}, []

for shot in shots:
    phys = DischargePhysics(load(shot))
    fits = {v: fit_mtanh(phys.ds, v, **FIT_KWARGS)[0] for v in ("Te", "Ti", "ne")}
    per_axis = {}
    for axis in AXES:
        var, kwarg, kind, unit, disp = AXES[axis]
        # width bounds come from the Boyle regime table, heights from TARGETS
        if kind == "width":
            key = "dTe" if var == "Te" else "dne"
            lo, hi = BOYLE_WIDTHS[REGIME][key]
        else:
            lo, hi = TARGETS["Ti_Te" if kind == "ti_te" else f"{var}_ped"]
        m0, slope, unit, disp = axis_response(phys, axis, fits)
        s_lo, s_hi = scale_for(m0, slope, lo), scale_for(m0, slope, hi)
        s_lo, s_hi = (min(s_lo, s_hi), max(s_lo, s_hi))
        ok = (np.isfinite(s_lo) and np.isfinite(s_hi)
              and SCALE_SANITY[0] <= s_lo and s_hi <= SCALE_SANITY[1])
        per_axis[axis] = {"nominal": m0, "unit": unit, "slope": slope,
                          "target": (lo, hi), "scale": (s_lo, s_hi), "ok": ok}
        reach_rows.append({"shot": shot, "axis": axis, **per_axis[axis]})
    bounds_per_shot[shot] = per_axis

hdr = (f"{'shot':>7} {'axis':<16} {'nominal':>10} {'unit':<6} {'target':>13} "
       f"{'slope':>7} {'need lo':>8} {'need hi':>8}  reach")
print(hdr); print("-" * len(hdr))
for r in reach_rows:
    lo, hi = r["target"]; sl, sh = r["scale"]
    mark = "ok" if r["ok"] else "OUT OF REACH"
    print(f"{r['shot']:>7} {r['axis']:<16} {r['nominal']:>10.3f} {r['unit']:<6} "
          f"{f'{lo}-{hi}':>13} {r['slope']:>7.3f} {sl:>8.2f} {sh:>8.2f}  {mark}")
print(f"\nreachable = both scale factors inside {SCALE_SANITY}; nothing beyond "
      f"that has been tested against cheaseBS, and the reshape limit is still "
      f"unmeasured")

### The scan box, in scale factors, ready for ScanStudy

Emitted per discharge, clipped to the sanity range so an unreachable target
produces a truncated box with a warning rather than a scale factor no
equilibrium will survive. Paste into `StudyConfig(bounds=...)`, or hand the dict
straight over.

In [ ]:
SG_BOUNDS = {}
for shot, per_axis in bounds_per_shot.items():
    box, notes = {}, []
    for axis, r in per_axis.items():
        sl, sh = r["scale"]
        if not (np.isfinite(sl) and np.isfinite(sh)):
            notes.append(f"{axis}: axis cannot move its metric — dropped")
            continue
        cl, ch = max(sl, SCALE_SANITY[0]), min(sh, SCALE_SANITY[1])
        if ch <= cl:
            notes.append(f"{axis}: target entirely out of reach — dropped")
            continue
        if (cl, ch) != (sl, sh):
            lo, hi = r["target"]
            reach_lo = r["nominal"] * (1 + r["slope"] * (cl - 1))
            reach_hi = r["nominal"] * (1 + r["slope"] * (ch - 1))
            notes.append(f"{axis}: clipped, reaches {reach_lo:.3g}-{reach_hi:.3g} "
                         f"{r['unit']} of {lo}-{hi}")
        box[axis] = (round(cl, 4), round(ch, 4))
    SG_BOUNDS[shot] = box
    print(f"{shot}: {json.dumps(box)}")
    for n in notes:
        print(f"    ! {n}")

with open("sg_bounds.json", "w") as f:
    json.dump({"regime": REGIME, "targets": TARGETS,
               "boyle_widths": BOYLE_WIDTHS[REGIME],
               "scale_sanity": list(SCALE_SANITY),
               "bounds": {str(k): v for k, v in SG_BOUNDS.items()}}, f, indent=1)
print("\nwritten: sg_bounds.json")

### Verify by eye before handing over

The box corners, drawn. This is the last check before the sparse grid takes
over: the curves should separate visibly, stay physical, and span the range the
table claims. A corner that overlaps the nominal is an axis contributing
nothing; a corner that goes negative or crosses another profile is a box that
needs pulling in regardless of what the arithmetic said.

In [ ]:
fig, axes_g = plt.subplots(2, len(shots), figsize=(3.7 * len(shots), 6),
                           squeeze=False)
for j, shot in enumerate(shots):
    phys = DischargePhysics(load(shot))
    fits = {v: fit_mtanh(phys.ds, v, **FIT_KWARGS)[0] for v in ("Te", "Ti", "ne")}
    x = np.asarray(phys.rhot.values)
    box = SG_BOUNDS[shot]
    for row, var in enumerate(("Te", "ne")):
        ax = axes_g[row][j]
        ax.plot(x, np.asarray(getattr(phys, var).values), "-", lw=2.0,
                color="k", label="nominal", zorder=3)
        for axis, (lo, hi) in box.items():
            v, kwarg, kind, _, _ = AXES[axis]
            if v != var:
                continue
            for s, ls in ((lo, "--"), (hi, ":")):
                q = phys.apply_mtanh_ped(var, fit=fits[v], **{kwarg: s},
                                         enforce_quasineutrality=True, qz=6.0)
                ax.plot(x, np.asarray(getattr(q, var).values), ls, lw=1.2,
                        label=f"{kwarg[6:]} {s:.2f}")
        ax.set_xlim(0.6, 1.0); ax.set_ylabel(var)
        if row == 0:
            ax.set_title(str(shot), fontsize=10)
        else:
            ax.set_xlabel("rho_tor")
        ax.tick_params(labelsize=7); ax.legend(fontsize=6)
fig.suptitle("scan box corners against nominal — verify before handing to the "
             "sparse grid")
plt.tight_layout(); plt.show()